# 07 — Data Collection: Scraping 2025 Legislative Results
Scrapes 2025 Portuguese legislative election results by district from Wikipedia using BeautifulSoup and requests.


In [ ]:
import pandas as pd

In [ ]:
# Lista dos distritos e regiões autónomas (como aparecem na Wikipédia em 2025)
distritos = [
    "Aveiro", "Beja", "Braga", "Bragança", "Castelo_Branco", "Coimbra", "Évora",
    "Faro", "Guarda", "Leiria", "Lisboa", "Portalegre", "Porto",
    "Santarém", "Setúbal", "Viana_do_Castelo", "Vila_Real", "Viseu",
    "Açores", "Madeira"
]

# Função para ler tabela com html5lib
def ler_tabela_wiki(distrito):
    url = f"https://pt.wikipedia.org/wiki/Resultados_das_eleicoes_legislativas_de_2025_em_{distrito}"
    try:
        tabelas = pd.read_html(url, flavor="html5lib")
        if tabelas:
            df = tabelas[0]
            df["Distrito"] = distrito.replace("_", " ")
            return df
    except Exception as e:
        print(f"Falhou em {distrito}: {e}")
    return None

# Juntar todas as tabelas
dfs = []
for dist in distritos:
    df_dist = ler_tabela_wiki(dist)
    if df_dist is not None:
        dfs.append(df_dist)

# Concatenar tudo
if dfs:
    resultados_2025 = pd.concat(dfs, ignore_index=True)
    display(resultados_2025.head())
    print("Total de linhas:", len(resultados_2025))

    # Guardar para CSV
    resultados_2025.to_csv("resultados_legislativas_2025_por_concelho.csv", index=False, encoding="utf-8-sig")
    print("✅ Resultados guardados em 'resultados_legislativas_2025_por_concelho.csv'")
else:
    print("❌ Nenhuma tabela foi extraída.")


In [ ]:
import requests
import pandas as pd

# Cabeçalho para fingir que és um browser real
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    )
}

# 🔗 Dicionário de distrito → URL (mete aqui os links reais!)
urls_por_distrito = {
    "Aveiro": "https://exemplo.site/eleicoes/2022/aveiro.html",
    "Beja": "https://exemplo.site/eleicoes/2022/beja.html",
    "Braga": "https://exemplo.site/eleicoes/2022/braga.html",
    # ... acrescenta o resto dos distritos
}

resultados = {}   # guardará um DataFrame por distrito

for distrito, url in urls_por_distrito.items():
    try:
        print(f"➡️  A ir buscar {distrito}…")
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        resp.encoding = "utf-8"   # essencial para acentos
        tabelas = pd.read_html(resp.text)  # devolve lista de DataFrames

        if tabelas:
            resultados[distrito] = tabelas[0]  # primeira tabela
            print(f"✅ {distrito}: {tabelas[0].shape}")
        else:
            print(f"⚠️ {distrito}: nenhuma tabela encontrada")

    except Exception as e:
        print(f"❌ Falhou em {distrito}: {e}")

# 👉 Podes juntar tudo num único DataFrame se as colunas forem iguais
if resultados:
    df_total = pd.concat(resultados.values(), keys=resultados.keys(), names=["Distrito"])
    df_total.reset_index(level="Distrito", inplace=True)
    print("🔎 Tabelas combinadas:", df_total.shape)
    display(df_total.head())
